In [1]:
%pip install -q -U ir_measures sentence_transformers

In [1]:
import os

os.environ["HF_TOKEN"] = input()

In [2]:
from datasets import load_dataset, Dataset
import os, json, time, random, pathlib
import requests
import numpy as np
import pandas as pd
import torch
from sentence_transformers import (
    SentenceTransformer,
    SentenceTransformerTrainer,
    SentenceTransformerTrainingArguments,
)
from sentence_transformers.evaluation import InformationRetrievalEvaluator
from sentence_transformers import CrossEncoder
from sentence_transformers.cross_encoder.evaluation import CrossEncoderRerankingEvaluator

/tmp/ipykernel_1155/3150954647.py:12: DeprecationWarning: Importing from 'sentence_transformers.evaluation' is deprecated and will be removed in a future version. Please use 'sentence_transformers.sentence_transformer.evaluation' instead.
  from sentence_transformers.evaluation import InformationRetrievalEvaluator


In [14]:
# dataset = load_dataset("pameydorke/arcanumchat-cross-platform-with-vague-queries-qwen-36-a3b-fp8")
dataset = load_dataset("pameydorke/arcanum-cross-platform-queries-synthetic") # old ds without removed documents for full unique documents ds

README.md:   0%|          | 0.00/585 [00:00<?, ?B/s]

data/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 2.93MB            

data/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

data/test-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  743kB            

data/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/4612 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1153 [00:00<?, ? examples/s]

In [19]:
traind_df = dataset['train'].to_pandas()
test_df = dataset['test'].to_pandas()
full_df = pd.concat([traind_df, test_df])

unique_docs_df = full_df.drop_duplicates(subset=['document_text'])
unique_docs_df = unique_docs_df.reset_index()
unique_docs_df = unique_docs_df.drop(columns=['query', 'query_type', 'difficulty', 'index'])

unique_docs_df = unique_docs_df.rename(columns={"document_text": "text", "document_link": "link", "document_title": "title"})

unique_docs_ds = Dataset.from_pandas(unique_docs_df)

unique_docs_ds

Dataset({
    features: ['title', 'text', 'link'],
    num_rows: 1153
})

In [6]:
DATA_URL = "https://raw.githubusercontent.com/ankorn/arcanumsearch/refs/heads/main/data/retriever_test_ds.json"
test_ds = requests.get(DATA_URL, timeout=30).json()

In [9]:
BASE_RETRIEVER_MODEL_ID = "BAAI/bge-m3"
FT_RETRIEVER_MODEL_ID = "pameydorke/arcanumchat-retriever-bge-m3-v2"

BASE_RERANKER_MODEL_ID = "BAAI/bge-reranker-v2-m3"
FT_RERANKER_MODEL_ID = "pameydorke/arcanumchat-reranker-bge-reranker-v2-m3-v2"

In [10]:

base_retriever_model = SentenceTransformer(BASE_RETRIEVER_MODEL_ID)
ft_retriever_model = SentenceTransformer(FT_RETRIEVER_MODEL_ID)

base_reranker_model = CrossEncoder(BASE_RERANKER_MODEL_ID, num_labels=1)
ft_reranker_model = CrossEncoder(FT_RERANKER_MODEL_ID, num_labels=1)

base_retriever_model.eval()
ft_retriever_model.eval()
base_reranker_model.eval()
ft_reranker_model.eval()

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/123 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/15.8k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/54.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/687 [00:00<?, ?B/s]

...

model.safetensors: reconstructing file:   0%|          |  0.00B / 2.27GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/393 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/409 [00:00<?, ?B/s]

CrossEncoder(
  (0): Transformer({'transformer_task': 'sequence-classification', 'modality_config': {'text': {'method': 'forward', 'method_output_name': 'logits'}}, 'module_output_name': 'scores', 'architecture': 'XLMRobertaForSequenceClassification'})
)

In [21]:
import hashlib

queries = {}
corpus = {}
relevant_docs = {}
url2text = {}

unique_docs = {}
for idx, example in enumerate(unique_docs_ds):
    doc_link = example["link"]
    doc_text = example["text"]
    url2text[doc_link] = doc_text

    if doc_text not in unique_docs:
        # doc_id = example["doc_id"]

        doc_id = hashlib.sha1(doc_text[:200].encode("utf-8")).hexdigest()[:16]
        unique_docs[doc_text] = doc_id
        corpus[doc_id] = doc_text

skipped_counter = 0
vague_counter = 0
for idx, example in enumerate(test_ds):
    # if example['is_vague']:
    #     vague_counter += 1
    #     continue

    query = example['query']
    url = example['url']

    if 'https://www.reddit.com' in url:
        url = url.replace('https://www.reddit.com', 'https://reddit.com')

    if url not in url2text:
        skipped_counter += 1
        continue

    doc_text = url2text[url]

    if doc_text not in unique_docs:
        skipped_counter += 1
        continue

    doc_id = unique_docs[doc_text]

    query_id = f"query_{idx}"
    queries[query_id] = query
    relevant_docs[query_id] = {doc_id}

print("Skipped:", skipped_counter)
print("Skipped vague:", vague_counter)

Skipped: 0
Skipped vague: 0


In [22]:
base_retriever_evaluator = InformationRetrievalEvaluator(
    queries=queries,
    corpus=corpus,
    relevant_docs=relevant_docs,
    name="arcanum-retriever-test",
)
ft_retriever_evaluator = InformationRetrievalEvaluator(
    queries=queries,
    corpus=corpus,
    relevant_docs=relevant_docs,
    name="arcanum-retriever-test",
)

base_retriever_results = base_retriever_evaluator(base_retriever_model)
ft_retriever_results = ft_retriever_evaluator(ft_retriever_model)

In [23]:
base_retriever_results

{'arcanum-retriever-test_cosine_accuracy@1': 0.65625,
 'arcanum-retriever-test_cosine_accuracy@3': 0.7864583333333334,
 'arcanum-retriever-test_cosine_accuracy@5': 0.8489583333333334,
 'arcanum-retriever-test_cosine_accuracy@10': 0.8697916666666666,
 'arcanum-retriever-test_cosine_precision@1': 0.65625,
 'arcanum-retriever-test_cosine_precision@3': 0.2621527777777778,
 'arcanum-retriever-test_cosine_precision@5': 0.16979166666666665,
 'arcanum-retriever-test_cosine_precision@10': 0.08697916666666665,
 'arcanum-retriever-test_cosine_recall@1': 0.65625,
 'arcanum-retriever-test_cosine_recall@3': 0.7864583333333334,
 'arcanum-retriever-test_cosine_recall@5': 0.8489583333333334,
 'arcanum-retriever-test_cosine_recall@10': 0.8697916666666666,
 'arcanum-retriever-test_cosine_ndcg@10': 0.7669381777985932,
 'arcanum-retriever-test_cosine_mrr@10': 0.7333519345238096,
 'arcanum-retriever-test_cosine_map@100': 0.7377885447490508}

In [24]:
ft_retriever_results

{'arcanum-retriever-test_cosine_accuracy@1': 0.5885416666666666,
 'arcanum-retriever-test_cosine_accuracy@3': 0.8020833333333334,
 'arcanum-retriever-test_cosine_accuracy@5': 0.8645833333333334,
 'arcanum-retriever-test_cosine_accuracy@10': 0.921875,
 'arcanum-retriever-test_cosine_precision@1': 0.5885416666666666,
 'arcanum-retriever-test_cosine_precision@3': 0.2673611111111111,
 'arcanum-retriever-test_cosine_precision@5': 0.1729166666666666,
 'arcanum-retriever-test_cosine_precision@10': 0.09218749999999999,
 'arcanum-retriever-test_cosine_recall@1': 0.5885416666666666,
 'arcanum-retriever-test_cosine_recall@3': 0.8020833333333334,
 'arcanum-retriever-test_cosine_recall@5': 0.8645833333333334,
 'arcanum-retriever-test_cosine_recall@10': 0.921875,
 'arcanum-retriever-test_cosine_ndcg@10': 0.7567313269459724,
 'arcanum-retriever-test_cosine_mrr@10': 0.7033895502645504,
 'arcanum-retriever-test_cosine_map@100': 0.707927695118174}

In [ ]:
# base_reranker_evaluator = CrossEncoderRerankingEvaluator(
#     samples=hard_eval_samples,
#     at_k=10,
#     name="val-hard-negatives",
#     batch_size=64,
#     show_progress_bar=True,
#     always_rerank_positives=True,
# )
# ft_reranker_evaluator = CrossEncoderRerankingEvaluator(
#     samples=hard_eval_samples,
#     at_k=10,
#     name="val-hard-negatives",
#     batch_size=64,
#     show_progress_bar=True,
#     always_rerank_positives=True,
# )

# base_reranker_results = base_reranker_evaluator(base_reranker_model)
# ft_reranker_results = ft_reranker_evaluator(ft_reranker_model)

In [25]:
import math
import torch
import numpy as np
import pandas as pd
from sentence_transformers import util


def encode_queries_and_corpus(model, queries, corpus, batch_size=64):
    qids = list(queries.keys())
    cids = list(corpus.keys())

    q_texts = [queries[q] for q in qids]
    c_texts = [corpus[c] for c in cids]

    q_emb = model.encode(
        q_texts,
        batch_size=batch_size,
        convert_to_tensor=True,
        normalize_embeddings=True,
        show_progress_bar=False,
    )
    c_emb = model.encode(
        c_texts,
        batch_size=batch_size,
        convert_to_tensor=True,
        normalize_embeddings=True,
        show_progress_bar=False,
    )
    return qids, cids, q_emb, c_emb

In [26]:
import torch
import numpy as np
from sentence_transformers import util
import ir_measures
from ir_measures import nDCG, RR, Recall
import hashlib

def per_query_metrics(model, reranker, queries, corpus, relevant_docs, k_values=(1, 3, 5, 10), k_retrieval=10):
    assert k_retrieval >= max(k_values)
    assert not model.training, "bi-encoder in train mode"

    qids, cids, q_emb, c_emb = encode_queries_and_corpus(model, queries, corpus)

    sim = util.cos_sim(q_emb, c_emb)
    initial_ranked_indices = torch.argsort(sim, dim=1, descending=True)[:, :k_retrieval]
    initial_ranked_cids = [[cids[i] for i in row] for row in initial_ranked_indices.tolist()]

    reranked_cids = []
    for i, qid in enumerate(queries):
        query_text = queries[qid]
        candidate_cids = initial_ranked_cids[i]
        candidate_docs = [corpus[cid] for cid in candidate_cids]

        pairs = [(query_text, doc) for doc in candidate_docs]
        scores = reranker.predict(pairs, convert_to_numpy=True)

        assert scores.ndim == 1, f"predict returned {scores.shape}"

        order = np.argsort(scores)[::-1]
        reranked_cids.append([candidate_cids[j] for j in order])

    # Build qrels and run dictionaries for ir_measures
    qrels = {qid: {doc_id: 1 for doc_id in rel_set} for qid, rel_set in relevant_docs.items()}
    run = {
        qid: {doc_id: float(len(reranked_cids[i]) - rank) for rank, doc_id in enumerate(reranked_cids[i])}
        for i, qid in enumerate(qids)
    }

    measures = []
    for k in k_values:
        measures.extend([Recall @ k, nDCG @ k])
    measures.append(RR)  # RR gives reciprocal rank per query; averaging yields MRR

    results = {qid: {} for qid in qids}
    for metric in ir_measures.iter_calc(measures, qrels, run):
        qid = metric.query_id
        name = str(metric.measure)
        if name == "RR":
            name = "mrr"  # Per-query RR is the MRR contribution
        results[qid][name] = metric.value

    return results

In [27]:
recall_metrics = ["R@3", "R@5", "R@10"]
ndcg_metrics = ["nDCG@10"]

def print_metrics(metrics, base_per_q, ft_per_q):
    for metric in metrics:
        base = np.array([base_per_q[q][metric] for q in queries])
        ft   = np.array([ft_per_q[q][metric]   for q in queries])
        delta = ft - base

        print("base mean:", base.mean())
        print("ft mean:  ", ft.mean())
        print("improved:", (delta > 0).sum())
        print("worse:   ", (delta < 0).sum())
        print("same:    ", (delta == 0).sum())

        rng = np.random.default_rng(0)
        n = len(delta)

        boot = np.array([
            delta[rng.integers(0, n, n)].mean()
            for _ in range(10_000)
        ])

        lo, hi = np.percentile(boot, [2.5, 97.5])
        print(f"mean delta = {delta.mean():.4f}")
        print(f"95% CI     = [{lo:.4f}, {hi:.4f}]", "\n")

In [28]:
base_per_q = per_query_metrics(base_retriever_model, base_reranker_model, queries, corpus, relevant_docs, k_retrieval=10)

ft_per_q = per_query_metrics(ft_retriever_model, ft_reranker_model, queries, corpus, relevant_docs, k_retrieval=10)

In [29]:
print_metrics(recall_metrics, base_per_q, ft_per_q)

base mean: 0.8125
ft mean:   0.8802083333333334
improved: 18
worse:    5
same:     169
mean delta = 0.0677
95% CI     = [0.0208, 0.1146] 

base mean: 0.8541666666666666
ft mean:   0.8958333333333334
improved: 15
worse:    7
same:     170
mean delta = 0.0417
95% CI     = [-0.0052, 0.0885] 

base mean: 0.8697916666666666
ft mean:   0.921875
improved: 14
worse:    4
same:     174
mean delta = 0.0521
95% CI     = [0.0104, 0.0990] 



In [30]:
print_metrics(ndcg_metrics, base_per_q, ft_per_q)

base mean: 0.7880848009509721
ft mean:   0.8487255110262363
improved: 37
worse:    15
same:     140
mean delta = 0.0606
95% CI     = [0.0225, 0.1001] 



In [ ]:
# base_ft_per_q = per_query_metrics(base_retriever_model, ft_reranker_model, queries, corpus, relevant_docs, k_retrieval=10)
# ft_base_per_q = per_query_metrics(ft_retriever_model,  base_reranker_model, queries, corpus, relevant_docs, k_retrieval=10)

In [31]:
base_per_q_20 = per_query_metrics(base_retriever_model, base_reranker_model, queries, corpus, relevant_docs, k_retrieval=20)
ft_per_q_20 = per_query_metrics(ft_retriever_model, ft_reranker_model, queries, corpus, relevant_docs, k_retrieval=20)

In [32]:
print_metrics(recall_metrics, base_per_q_20, ft_per_q_20)

base mean: 0.8177083333333334
ft mean:   0.8854166666666666
improved: 15
worse:    2
same:     175
mean delta = 0.0677
95% CI     = [0.0312, 0.1094] 

base mean: 0.8645833333333334
ft mean:   0.921875
improved: 15
worse:    4
same:     173
mean delta = 0.0573
95% CI     = [0.0156, 0.0990] 

base mean: 0.8958333333333334
ft mean:   0.9583333333333334
improved: 13
worse:    1
same:     178
mean delta = 0.0625
95% CI     = [0.0260, 0.0990] 



In [33]:
print_metrics(ndcg_metrics, base_per_q_20, ft_per_q_20)

base mean: 0.7929369585533795
ft mean:   0.8705032704103971
improved: 41
worse:    16
same:     135
mean delta = 0.0776
95% CI     = [0.0406, 0.1170] 



In [34]:
rows = []
for qid in queries:
    row = {"qid": qid, "query": queries[qid]}
    for k, v in base_per_q[qid].items():
        row[f"base_{k}"] = v
    for k, v in ft_per_q[qid].items():
        row[f"ft_{k}"] = v
    rows.append(row)

df = pd.DataFrame(rows)
df["delta_ndcg@10"] = df["ft_nDCG@10"] - df["base_nDCG@10"]
df = df.sort_values("delta_ndcg@10")

# Queries where fine-tuning hurt most
print(df[["qid", "query", "base_nDCG@10", "ft_nDCG@10", "delta_ndcg@10"]].head(20))

           qid                                   query  base_nDCG@10  \
164  query_164                      sence hiiden spell      1.000000   
41    query_41                         flickering icon      1.000000   
153  query_153            simple shop dealer from Boil      0.630930   
23    query_23                              find elves      1.000000   
36    query_36                                     UAP      0.500000   
132  query_132                   stonecutter clan cave      1.000000   
25    query_25                              magic book      0.430677   
51    query_51        cathedral ruins near Roseborough      1.000000   
101  query_101  Tarnat guards attack after killing npc      1.000000   
121  query_121               will clan mines kill mobs      1.000000   
79    query_79                             pick poxket      1.000000   
137  query_137                    sniper rifgle schema      0.630930   
15    query_15                  bow mastr trainee find      0.63

In [35]:
# Build a list of (qid, base, ft, delta) for recall@5
rows = []
for qid in queries:
    b = base_per_q[qid]["R@10"]
    f = ft_per_q[qid]["R@10"]
    rows.append((qid, b, f, f - b))

# Keep only the ones that got worse
worse = [r for r in rows if r[3] < 0]
worse.sort(key=lambda r: r[3])  # most negative first

print(f"Total worse on recall@10: {len(worse)}")
print("=" * 80)

for qid, b, f, d in worse:
    # print(f"qid:     {qid}")
    print(f"query:   {queries[qid]}")
    # print(f"base:    {b:.3f}   ft: {f:.3f}   delta: {d:+.3f}")

    rel_docs = relevant_docs[qid]
    # print(f"relevant doc ids: {rel_docs}")

    # Show the text of the relevant docs (truncated)
    for doc_id in rel_docs:
        text = corpus[doc_id]
        # print(f"  [{doc_id}] {text[:300]}...")

    # If you want to see where the fine-tuned model ranked the gold doc,
    # you need the ranked list. See helper below.
    print("-" * 80)

Total worse on recall@10: 4
query:   magic book
--------------------------------------------------------------------------------
query:   UAP
--------------------------------------------------------------------------------
query:   simple shop dealer from Boil
--------------------------------------------------------------------------------
query:   sence hiiden spell
--------------------------------------------------------------------------------
